In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Lendo as tabelas do catálogo
df_users = spark.table("workspace.default.users_yt")
df_posts = spark.table("workspace.default.posts_creator")

# Criando Views Temporárias para usar SQL puro também
df_users.createOrReplaceTempView("users_yt")
df_posts.createOrReplaceTempView("posts_creator")

In [0]:
# Top 3 Posts por Likes Últimos 6 meses

# --- PASSO 1: DESCOBRIR A DATA DE REFERÊNCIA ---
# Em sustentação, é comum os dados estarem atrasados. 
# Em vez de usar a data de "hoje" do sistema, buscamos a data do último registro no banco.

# Criamos uma coluna temporária com a data legível para achar o máximo
max_date_df = df_posts.select(
    F.max(F.from_unixtime(F.col("published_at"))).cast("date")
).alias("max_date")

# Extraímos esse valor para uma variável Python para usar no cálculo de meses
# O .collect()[0][0] acessa o valor dentro da primeira linha e primeira coluna do resultado
latest_date_available = max_date_df.collect()[0][0]

# Calculamos o corte: 6 meses antes da data mais recente encontrada
# Usamos F.lit() para transformar a variável Python em uma coluna constante no Spark
date_limit = F.add_months(F.lit(latest_date_available), -6)

# --- PASSO 2: PREPARAÇÃO DOS DADOS ---
df_posts_fixed = df_posts.withColumn(
    "published_at_ts",
    F.from_unixtime(F.col("published_at")).cast("date")
)


# --- PASSO 3: DEFINIÇÃO DA REGRA DE NEGÓCIO (JANELA) ---

# A Window (janela) permite olhar para grupos de dados sem "sumir" com as linhas (diferente do GroupBy)
window_likes = Window.partitionBy("user_id").orderBy(F.col("likes").desc())


# --- PASSO 4: PROCESSAMENTO FINAL ---

top_3_likes = (
    # Unimos a tabela de posts com a de usuários para pegar informações de perfil
    # yt_user: campo de ligação no post / user_id: campo de ligação no usuário
    df_posts_fixed.join(df_users, df_posts_fixed.yt_user == df_users.user_id)
    
    # Filtro Dinâmico: Só o que aconteceu nos últimos 6 meses a partir do dado mais novo
    .filter(F.col("published_at_ts") >= date_limit)
    
    # Criamos o ranking: row_number() atribui 1 para o post com mais likes de cada user_id
    .withColumn("rank", F.row_number().over(window_likes))
    
    # Filtramos para manter apenas os medalhistas (Top 3)
    .filter(F.col("rank") <= 3)
    
    # Seleção limpa para o relatório final
    .select(
        "user_id", 
        "title", 
        "likes", 
        "rank", 
        "published_at_ts" # Importante para a sustentação validar o período
    )
)

# --- PASSO 5: SAÍDA ---
# Exibe os dados. Em produção, isso seria substituído por um .write.save()
top_3_likes.show()

In [0]:
#Top 3 Posts por Likes (Últimos 6 meses)from pyspark.sql import functions as F


# --- CONFIGURAÇÃO DE DATAS ---
# 1. Pegamos a data máxima real dos seus dados (15/04/2024)
max_date_val = df_posts.select(F.max(F.from_unixtime(F.col("published_at"))).cast("date")).collect()[0][0]

# 2. Calculamos o corte (6 meses antes de 15/04/2024 -> 15/10/2023)
cutoff_date = F.add_months(F.lit(max_date_val), -6)

# --- TRATAMENTO DOS POSTS ---
# Convertendo o tempo (Usando epoch_in_ms=False pois o teste confirmou segundos)
df_posts_clean = df_posts.withColumn(
    "published_at_ts", 
    F.from_unixtime(F.col("published_at")).cast("date")
)

# --- REGRA DE NEGÓCIO (TOP 3) ---
# Definimos a janela: agrupa por usuário, ordena por likes (maior primeiro)
window_spec = Window.partitionBy("user_id").orderBy(F.col("likes").desc())

# --- PROCESSAMENTO FINAL ---
df_final = (
    df_posts_clean
    # Unindo as tabelas (O teste confirmou que yt_user e user_id batem)
    .join(df_users, df_posts_clean.yt_user == df_users.user_id, how="inner")
    # Aplicando o filtro baseado na data máxima dos dados
    .filter(F.col("published_at_ts") >= cutoff_date)
    # Criando o ranking
    .withColumn("rank", F.row_number().over(window_spec))
    # Pegando apenas os 3 primeiros
    .filter(F.col("rank") <= 3)
    .select("user_id", "title", "likes", "published_at_ts", "rank")
)

# --- EXIBIÇÃO ---
print(f"Relatório gerado considerando o período: {max_date_val} voltando até 6 meses.")
df_final.show(truncate=False)


In [0]:
# --- RELATÓRIO: TOP 3 POSTS POR VIEWS (DATA DINÂMICA) ---

# --- RELATÓRIO: TOP 3 POSTS POR VIEWS (DINÂMICO E SEGURO) ---

# Primeiro, extraímos a data máxima para fora da query.
# Isso garante que o valor seja injetado como um texto simples, evitando erros de tipo no SQL.
max_date_val = spark.sql("SELECT MAX(CAST(FROM_UNIXTIME(published_at) AS DATE)) FROM posts_creator").collect()[0][0]

# Se por algum motivo a tabela estiver vazia, interrompe aqui para evitar erro
if max_date_val is None:
    print("ERRO DE SUSTENTAÇÃO: A tabela 'posts_creator' está vazia ou as datas são nulas.")
else:
    # Definimos a query injetando a data máxima diretamente (F-String)
    # Note que usamos a data máxima encontrada (2024-04-15) para subtrair 6 meses
    query_top_views = f"""
    SELECT * FROM (
        SELECT 
            u.user_id, 
            p.title, 
            p.views,
            CAST(FROM_UNIXTIME(p.published_at) AS DATE) as data_publicacao,
            ROW_NUMBER() OVER (PARTITION BY u.user_id ORDER BY p.views DESC) as rank
        FROM posts_creator p
        INNER JOIN users_yt u ON p.yt_user = u.user_id
        
        -- LÓGICA DE CORTE: 
        -- Filtramos 6 meses antes da data máxima encontrada nos dados ('{max_date_val}')
        WHERE CAST(FROM_UNIXTIME(p.published_at) AS DATE) >= ADD_MONTHS(DATE '{max_date_val}', -6)
    ) 
    WHERE rank <= 3
    ORDER BY user_id, views DESC
    """

    # Executa e exibe
    df_top_views = spark.sql(query_top_views)
    
    if df_top_views.count() == 0:
        print(f"AVISO: O Join ou o filtro de data resultou em 0 linhas para a data máxima {max_date_val}")
    else:
        display(df_top_views)

In [0]:
#Creators em 'posts_creator' ausentes em 'users_yt'

# O objetivo aqui é encontrar quem está postando vídeos (df_posts) 
# mas não consta na nossa tabela oficial de usuários (df_users).

mismatch_creators = df_posts.join(
    df_users, 
    # Comparamos o ID do usuário que fez o post com o ID da tabela de cadastro
    df_posts.yt_user == df_users.user_id, 
    
    # O segredo está aqui: "left_anti" (Join Anti-Esquerda)
    # Ele retorna APENAS as linhas da tabela da esquerda (posts) 
    # que NÃO possuem uma correspondência na tabela da direita (users).
    how="left_anti"
).select(
    # Selecionamos apenas a coluna do usuário problemático
    F.col("yt_user")
).distinct() # Removemos duplicatas: se o user postou 10 vezes, ele aparece só 1 vez aqui.

# --- EXIBIÇÃO DO RESULTADO ---

print("ALERTA DE SUSTENTAÇÃO: Creators presentes em posts mas ausentes no cadastro de usuários:")
# Se o .show() retornar nomes, esses usuários estão "invisíveis" nos relatórios principais
mismatch_creators.show()

In [0]:
#Análise Mensal
# --- RELATÓRIO DE VOLUMETRIA: FREQUÊNCIA DE POSTAGEM POR MÊS ---

# 1. PREPARAÇÃO DA BASE (Extração de competência)
# Criamos uma coluna 'month' para agrupar os posts por período (Ex: '2023-10').
# Nota de Sustentação: O tempo está em segundos (epoch), por isso não dividimos por 1000.
df_monthly_base = df_posts.withColumn(
    "month", 
    F.date_format(F.from_unixtime(F.col("published_at")), "yyyy-MM")
)

# 2. CONSTRUÇÃO DA MATRIZ PIVOTADA (Transformação de Linhas em Colunas)
df_pivot_analysis = (
    df_monthly_base
    .groupBy("yt_user")           # Agrupamos por criador
    .pivot("month")               # Cada mês único vira uma coluna no relatório
    .agg(F.count("title"))        # Contamos quantos títulos (posts) existem por mês
    
    # TRATAMENTO DE DADOS VAZIOS:
    # Se um criador não postou em um mês específico, o Spark retorna 'null'. 
    # O .fillna(0) garante que o relatório mostre '0', facilitando cálculos futuros.
    .fillna(0) 
    
    # Padronização de nome de coluna para manter consistência com a tabela de usuários
    .withColumnRenamed("yt_user", "user_id")
)

# --- EXIBIÇÃO ---
# O resultado mostrará uma linha por usuário e uma coluna para cada mês encontrado.
print("Análise de Performance Mensal (Quantidade de publicações):")
display(df_pivot_analysis)

In [0]:
###################################################Salvando os Resultados em Tabelas Delta################################################ (EXTRA)
# 1. Definindo o caminho e nome da tabela de performance mensal
df_top_views = spark.sql(query_top_views) 

# 2. Definindo o caminho e nome da tabela de performance mensal
target_analytics_table = "workspace.default.analytics_creators_monthly"

# Salvando o DataFrame Pivotado
(df_pivot_analysis.write
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(target_analytics_table))

# 3. Criando a tabela de Ranking (Unindo os dois DataFrames)
# Agora usamos df_top_views (DataFrame) em vez de query_top_views (String)
df_rankings = top_3_likes.unionByName(df_top_views, allowMissingColumns=True)

(df_rankings.write
    .mode("overwrite")
    .option("mergeSchema", "true") # Boa prática adicionar aqui também
    .saveAsTable("workspace.default.analytics_creators_ranking"))

print(f"✅ Tabelas de analytics criadas com sucesso no catálogo.")